# 🎮 Full Fine-tuning (Game-optimized / KcELECTRA)

게임 음성채팅 환경에 최적화된 Full Fine-tuning 노트북입니다.

## 📋 특징
- **모델**: `beomi/KcELECTRA-base-v2022` (한국어 최적화)
- **메트릭**: `abuse_recall` (악플 탐지율)
- **방식**: Full Fine-tuning (모든 파라미터 학습)

## ⚠️ 주의사항
- 모든 파라미터를 학습하므로 **GPU 메모리 많이 사용**
- LoRA 대비 **학습 시간 더 소요**
- L40S GPU (48GB) 환경 권장

## 1. 환경 설정 및 라이브러리 임포트

In [ ]:
import os
import torch
import pandas as pd
import numpy as np
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from sklearn.metrics import precision_recall_fscore_support, label_ranking_average_precision_score
from datasets import Dataset
import warnings
warnings.filterwarnings('ignore')

print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. 하이퍼파라미터 설정

Full Fine-tuning 설정:
- **LR=2e-5**: Full FT는 낮은 LR 사용
- **batch_size=16**: 메모리 절약을 위해 작게

In [ ]:
MODEL_NAME = "beomi/KcELECTRA-base-v2022"
OUTPUT_DIR = "./output_game_full"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 학습 하이퍼파라미터
EPOCHS = 5
BATCH_SIZE = 16  # Full FT는 메모리 많이 사용
LEARNING_RATE = 2e-5  # Full FT는 낮은 LR
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01
MAX_LENGTH = 128

LABEL_NAMES = ["여성/가족", "남성", "성소수자", "인종/국적", "연령",
               "지역", "종교", "기타 혐오", "악플/욕설", "clean"]
NUM_LABELS = len(LABEL_NAMES)

print(f"Device: {DEVICE}")
print(f"Model: {MODEL_NAME}")

## 3. 데이터 로드

In [ ]:
TRAIN_PATH = "../3_UnSmile_Correction/unsmile_train_corrected.tsv"
VALID_PATH = "../3_UnSmile_Correction/unsmile_valid_corrected.tsv"

train_df = pd.read_csv(TRAIN_PATH, sep='\t', encoding='utf-8')
valid_df = pd.read_csv(VALID_PATH, sep='\t', encoding='utf-8')

print(f"Train 데이터: {len(train_df)}건")
print(f"Valid 데이터: {len(valid_df)}건")

## 4. 토크나이저 및 전처리

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess_function(examples):
    tokenized = tokenizer(
        examples['문장'],
        padding='max_length',
        truncation=True,
        max_length=MAX_LENGTH
    )
    labels = [[float(examples[col][i]) for col in LABEL_NAMES] 
              for i in range(len(examples['문장']))]
    tokenized['labels'] = labels
    return tokenized

train_dataset = Dataset.from_pandas(train_df).map(
    preprocess_function, batched=True, remove_columns=train_df.columns.tolist())
valid_dataset = Dataset.from_pandas(valid_df).map(
    preprocess_function, batched=True, remove_columns=valid_df.columns.tolist())

print("전처리 완료!")

## 5. 모델 로드 (Full Fine-tuning)

LoRA와 달리 **모든 파라미터가 학습**됩니다.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification"
)
model.config.id2label = {i: l for i, l in enumerate(LABEL_NAMES)}
model.config.label2id = {l: i for i, l in enumerate(LABEL_NAMES)}
model = model.to(DEVICE)

# 전체 파라미터 수 확인
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,} (100%)")

## 6. 평가 메트릭 (게임 환경 최적화)

In [ ]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    probs = torch.sigmoid(torch.tensor(predictions)).numpy()
    preds = (probs > 0.5).astype(int)
    labels_int = labels.astype(int)
    
    # 악플/욕설 (index 8)
    abuse_p, abuse_r, abuse_f1, _ = precision_recall_fscore_support(
        labels_int[:,8], preds[:,8], average='binary', zero_division=0
    )
    
    # Clean (index 9)
    clean_p, clean_r, clean_f1, _ = precision_recall_fscore_support(
        labels_int[:,9], preds[:,9], average='binary', zero_division=0
    )
    
    lrap = label_ranking_average_precision_score(labels, predictions)
    
    return {
        'lrap': lrap,
        'abuse_recall': abuse_r,
        'abuse_f1': abuse_f1,
        'clean_recall': clean_r,
        'clean_f1': clean_f1,
    }

## 7. 학습 설정 및 실행

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="abuse_recall",
    greater_is_better=True,
    logging_steps=50,
    save_total_limit=2,
    report_to="none",
    fp16=True,
    gradient_accumulation_steps=2,  # 효과적 batch_size=32
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

In [ ]:
print("⚠️ Full Fine-tuning 시작 (시간 소요)...")
trainer.train()
print("학습 완료!")

## 8. 모델 저장

In [ ]:
model.save_pretrained(f"{OUTPUT_DIR}/best_model")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/best_model")
print(f"모델 저장 완료: {OUTPUT_DIR}/best_model")

## 9. 최종 평가

In [ ]:
print("="*60)
print("📊 최종 평가 결과 (Full FT Game-optimized)")
print("="*60)

eval_results = trainer.evaluate()
for key, value in eval_results.items():
    print(f"  {key}: {value:.4f}")

with open(f"{OUTPUT_DIR}/results.txt", 'w', encoding='utf-8') as f:
    f.write("=== Full FT Game-optimized (KcELECTRA) ===\n")
    f.write(f"Model: {MODEL_NAME}\n")
    f.write("Method: Full Fine-tuning\n")
    for k, v in eval_results.items():
        f.write(f"{k}: {v:.4f}\n")

print("="*60)
print("✅ 완료!")